In [14]:
import duckdb
import pandas as pd
import requests

In [15]:
credentials = {
    "client_id": "230098",
    "client_secret": "57ff9f08daffc8d77771e03bae7c380eed00d7b0",
    "refresh_token": "5d06f94ff6551fe71a0c8ec609a90b152f0997f9",
}

In [16]:
DB_PATH = "wynne_strava.db"

In [17]:
def connect_to_strava(credentials):
    """Exchange a refresh token for an access token and return auth headers."""
    response = requests.post(
        "https://www.strava.com/oauth/token",
        data={**credentials, "grant_type": "refresh_token"},
    )
    response.raise_for_status()
    access_token = response.json()["access_token"]
    return {"Authorization": f"Bearer {access_token}"}

In [18]:
def fetch_activities(headers):
    """Fetch athlete activities from Strava and return as a DataFrame."""
    response = requests.get(
        "https://www.strava.com/api/v3/athlete/activities",
        headers=headers,
        params={"per_page": 200, "page": 1},
    )
    response.raise_for_status()
    return pd.json_normalize(response.json())

In [19]:
def create_database(db_path):
    """Create (or open) the DuckDB database and return the connection."""
    conn = duckdb.connect(db_path)
    return conn

In [20]:
def create_wynne_activities_table(conn, activities_df):
    """Create the wynne_activities table from a DataFrame of Strava activities."""
    conn.sql("CREATE TABLE IF NOT EXISTS wynne_activities AS SELECT * FROM activities_df")

In [ ]:
headers = connect_to_strava(credentials)
activities_df = fetch_activities(headers)

conn = create_database(DB_PATH)
create_wynne_activities_table(conn, activities_df)

In [23]:
conn = duckdb.connect(DB_PATH)

In [24]:
conn.sql("SELECT * FROM wynne_activitieS ORDER BY start_date_local DESC LIMIT 5")

┌────────────────┬───────────────┬──────────┬─────────────┬──────────────┬──────────────────────┬─────────┬────────────┬──────────────┬─────────────┬─────────────┬──────────────────────┬──────────────────────┬─────────────────────────────┬────────────┬───────────────┬────────────────┬──────────────────┬───────────────────┬─────────────┬───────────────┬───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┬────────────┬─────────┬─────────┬─────────────────────────┬─────────────────────────┬───────────────┬───────────┬───────────────┬───────────────────┬───────────────────────────────┬───────────┬──────────┬─────────────┬───────────────┬────────────────────────────────────────────────────────────┬───────────────────┬──────────┬───────────────────┬────────────┬────────────┬────────────────────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [26]:
conn.close()